In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="Qwen/Qwen3-8B-MLX-4bit",
    base_url="http://10.195.19.15:8000/v1",
    api_key="dummy",
    temperature=0,
    max_tokens=8192,
)

In [4]:
response = model.invoke("/no_think Say 'stack is alive' if you can hear me.")
print(response.content)



Stack is alive.


## Step 1: load one CSV and inspect it manually

No agent yet — just looking at the raw data so we know its shape before
deciding what the LLM should see (schema + preview only, never the full table).

In [5]:
import pandas as pd

df = pd.read_csv("bazaar_books/caravan_accounts.csv")

print(df.shape)
df.dtypes

(176, 9)


Realm                   str
Guild_Name              str
Year                  int64
Quarter               int64
Operating_Income    float64
EBITDA              float64
Tax                 float64
Net_Income          float64
GOGS                float64
dtype: object

In [6]:
df.head()

,Realm,Guild_Name,Year,Quarter,Operating_Income,EBITDA,Tax,Net_Income,GOGS
0,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,1,21904.87,26090.31,5855.70,16049.17,75668.54
1,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,2,17681.07,23523.18,3967.18,13713.89,62357.84
2,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,3,20009.92,31315.08,4861.25,15148.67,85116.94
3,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,4,14784.45,22956.75,3998.11,10786.34,41932.53
4,Oasis of Whispering Sands,Djinn-Forged Ironworks,718,1,13575.22,16403.08,3558.74,10016.48,35353.81


## Step 2: build the schema-and-preview system prompt

The LLM never sees the full DataFrame. Instead it sees, once, at the start:
- the schema (column names + dtypes) as a markdown table
- a small preview (first N rows) as markdown-KV (key: value pairs) — chosen
  over a markdown table because research shows markdown-KV gives higher
  comprehension accuracy for small models, at the cost of more tokens
  (acceptable here since it's only a handful of rows).

In [7]:
def build_schema_table(df: pd.DataFrame) -> str:
    """Render column names + dtypes as a markdown table.

    Args:
        df: the DataFrame to describe.

    Returns:
        A markdown table string, one row per column: "column | dtype".
    """
    lines = ["| column | dtype |", "|---|---|"]
    for column_name, dtype in df.dtypes.items():
        lines.append(f"| {column_name} | {dtype} |")
    return "\n".join(lines)


def build_preview_kv(df: pd.DataFrame, n_rows: int = 5) -> str:
    """Render the first n_rows of a DataFrame as markdown-KV blocks.

    Each row becomes a block of "column: value" lines separated by "---".
    Chosen over a markdown table for better small-model comprehension.

    Args:
        df: the DataFrame to preview.
        n_rows: how many rows from the top to include.

    Returns:
        A markdown-KV formatted string.
    """
    blocks = []
    for _, row in df.head(n_rows).iterrows():
        lines = [f"{column_name}: {value}" for column_name, value in row.items()]
        blocks.append("\n".join(lines))
    return "\n---\n".join(blocks)


print(build_schema_table(df))
print()
print(build_preview_kv(df))

| column | dtype |
|---|---|
| Realm | str |
| Guild_Name | str |
| Year | int64 |
| Quarter | int64 |
| Operating_Income | float64 |
| EBITDA | float64 |
| Tax | float64 |
| Net_Income | float64 |
| GOGS | float64 |

Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 1
Operating_Income: 21904.87
EBITDA: 26090.31
Tax: 5855.7
Net_Income: 16049.17
GOGS: 75668.54
---
Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 2
Operating_Income: 17681.07
EBITDA: 23523.18
Tax: 3967.18
Net_Income: 13713.89
GOGS: 62357.84
---
Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 3
Operating_Income: 20009.92
EBITDA: 31315.08
Tax: 4861.25
Net_Income: 15148.67
GOGS: 85116.94
---
Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 4
Operating_Income: 14784.45
EBITDA: 22956.75
Tax: 3998.11
Net_Income: 10786.34
GOGS: 41932.53
---
Realm: Oasis of Whispering Sands
Guild_

In [8]:
SYSTEM_PROMPT_TEMPLATE = """\
You are a financial analytics assistant. You answer questions about a single
pandas DataFrame called `df`, which is already loaded in your execution
environment — you never need to load or recreate it.

You do not have the full table in front of you. You have only the schema
and a small preview below. To answer any question that needs real numbers,
you must call the `execute_python_code` tool with pandas code that operates
on `df` and returns the result. Never guess numeric values — always compute
them via the tool.

## Schema

{schema_table}

## Preview (first {n_preview_rows} rows)

{preview_kv}

## How to answer

- For small talk ("hello", "thank you") — respond directly, do not call the tool.
- For any question needing numbers from the data — write pandas code against
  `df` and call `execute_python_code`. Do not answer from memory or from the
  preview above; the preview is only a sample, not the full data.
"""


def build_system_prompt(df: pd.DataFrame, n_preview_rows: int = 5) -> str:
    """Build the full system prompt for the analytics agent.

    Args:
        df: the DataFrame the agent will answer questions about.
        n_preview_rows: how many rows to include in the preview section.

    Returns:
        The rendered system prompt string.
    """
    return SYSTEM_PROMPT_TEMPLATE.format(
        schema_table=build_schema_table(df),
        n_preview_rows=n_preview_rows,
        preview_kv=build_preview_kv(df, n_preview_rows),
    )


system_prompt = build_system_prompt(df)
print(system_prompt)

You are a financial analytics assistant. You answer questions about a single
pandas DataFrame called `df`, which is already loaded in your execution
environment — you never need to load or recreate it.

You do not have the full table in front of you. You have only the schema
and a small preview below. To answer any question that needs real numbers,
you must call the `execute_python_code` tool with pandas code that operates
on `df` and returns the result. Never guess numeric values — always compute
them via the tool.

## Schema

| column | dtype |
|---|---|
| Realm | str |
| Guild_Name | str |
| Year | int64 |
| Quarter | int64 |
| Operating_Income | float64 |
| EBITDA | float64 |
| Tax | float64 |
| Net_Income | float64 |
| GOGS | float64 |

## Preview (first 5 rows)

Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 1
Operating_Income: 21904.87
EBITDA: 26090.31
Tax: 5855.7
Net_Income: 16049.17
GOGS: 75668.54
---
Realm: Oasis of Whispering Sands
Gui

## Step 3: the `execute_python_code` tool

Same idea as the old OpenAI Code Interpreter workflow this project replaces:
the LLM writes pandas code that ends in `print(...)`, the tool runs that
code against the real `df` with `exec()`, captures whatever was printed to
stdout, and returns it as text. If the code raises an exception, we don't
crash — we return the error message back to the agent so it can see what
went wrong and try again with corrected code.

In [9]:
import contextlib
import io

from langchain_core.tools import tool

MAX_TOOL_OUTPUT_CHARS = 4000


@tool
def execute_python_code(code: str) -> str:
    """Run pandas code against the loaded financial DataFrame `df` and return its printed output.

    The DataFrame `df` and the `pd` (pandas) module are already available —
    do not try to import pandas or load/recreate `df` yourself.

    Your code MUST call print(...) on whatever value answers the question.
    Anything not printed is lost — this tool only returns what was printed.

    Print only what you need to answer the question (a single value, a small
    aggregate, a short table) — do not print the entire DataFrame. Output is
    truncated past a length limit, since `df` may be much larger in real use.

    Args:
        code: a snippet of Python/pandas code, e.g.
            "print(df.groupby('Realm')['Net_Income'].sum().idxmax())"

    Returns:
        Everything the code printed to stdout, as a single string (truncated
        if too long, with a note telling you to narrow your query). If the
        code raised an exception instead, returns an "Error: ..." message
        describing what went wrong, so you can fix the code and try again.
    """
    namespace = {"df": df, "pd": pd}
    stdout_buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_buffer):
            exec(code, namespace)
    except Exception as e:
        return f"Error: {e}"

    output = stdout_buffer.getvalue()
    if not output:
        return "Code ran without errors but printed nothing. Use print() to show a result."

    if len(output) > MAX_TOOL_OUTPUT_CHARS:
        truncated = output[:MAX_TOOL_OUTPUT_CHARS]
        return (
            f"{truncated}\n"
            f"...\n"
            f"[Output truncated at {MAX_TOOL_OUTPUT_CHARS} characters — your code printed too "
            f"much. Narrow your query: filter rows first, aggregate instead of printing raw "
            f"rows, or use .head()/.describe() instead of printing the whole result.]"
        )
    return output


In [8]:

# quick manual check — no LLM involved, just calling the tool function directly
print(execute_python_code.invoke({"code": "print(df.groupby('Realm')['Net_Income'].sum().idxmax())"}))

Garden of the Midnight Rose



### Guard test: simulate a huge print (no LLM involved)

Our real dataset is only 176 rows, too small to actually trigger the
truncation naturally. This manually forces a large output (repeating text)
to confirm the guard kicks in correctly before we ever hit it with a real
large table.

In [10]:
huge_print_code = "for i in range(2000): print(f'row {i}: some financial data goes here')"
result = execute_python_code.invoke({"code": huge_print_code})

print(f"Returned length: {len(result)} chars (limit: {MAX_TOOL_OUTPUT_CHARS})")
print("Ends with:")
print(result[-300:])

Returned length: 4215 chars (limit: 4000)
Ends with:
 some financial data goes here
row 104: some financial data goes here
row 105: some f
...
[Output truncated at 4000 characters — your code printed too much. Narrow your query: filter rows first, aggregate instead of printing raw rows, or use .head()/.describe() instead of printing the whole result.]


## Step 4: assemble the agent

`create_agent` wires together the model, the tool, and our system prompt
into a ReAct loop (agent node ↔ tools node, repeating until the model
stops calling tools). This cell only builds the agent object — no LLM
call happens yet. The first real call is the next step.

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[execute_python_code],
    system_prompt=system_prompt,
)

## Step 5: first end-to-end question

This is the first real call to the LLM through the agent. We ask a
question adapted from the sample queries in the README — instead of
"country" / 2023, we use our synthetic setting's "realm" / year 718.
We print every message in the result so we can see the full ReAct loop:
the AI message with a tool call, the ToolMessage with the tool's output,
and the final AI answer.

In [12]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Which company made the most profit?"}]}
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Which company made the most profit?
================================== Ai Message ==================================
Tool Calls:
  execute_python_code (5f619a5c-e5c7-4ec5-a625-051d855ffe34)
 Call ID: 5f619a5c-e5c7-4ec5-a625-051d855ffe34
  Args:
    code: print(df.groupby('Guild_Name')['Net_Income'].sum().idxmax())
================================= Tool Message =================================
Name: execute_python_code

Vizier's Ledger & Ore Co.

================================== Ai Message ==================================



The company that made the most profit is **Vizier's Ledger & Ore Co.**, based on the highest total Net Income across all quarters.


### Debug: inspect the last AI message raw

`pretty_print()` only shows `.content`. If content is empty, the real
information is elsewhere — either `tool_calls` (model decided to call a
tool but message rendering hid it) or `response_metadata` (e.g. a
`finish_reason: "length"` meaning the model hit `max_tokens` before
producing visible output, likely spent on hidden `<think>...</think>`
reasoning tokens Qwen3 emits by default).

In [11]:
last_ai_message = [m for m in result["messages"] if m.type == "ai"][-1]

print("content:", repr(last_ai_message.content))
print("tool_calls:", last_ai_message.tool_calls)
print("response_metadata:", last_ai_message.response_metadata)

content: "\n\nThe company that made the most profit is **Vizier's Ledger & Ore Co.** This was determined by summing the `Net_Income` for each guild and identifying the one with the highest total."
tool_calls: []
response_metadata: {'token_usage': {'completion_tokens': 365, 'prompt_tokens': 1149, 'total_tokens': 1514, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 1091}}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-8B-MLX-4bit', 'system_fingerprint': '0.31.2-0.31.1-macOS-26.3.1-arm64-arm-64bit-applegpu_g13s', 'id': 'chatcmpl-90689088-7d03-4ea6-82f4-8eb5e6cc81a9', 'finish_reason': 'stop', 'logprobs': None}


## Step 6: test against reference questions (no dedicated tools yet)

Questions adapted from the legacy Forvis Mazars assistant's sample queries
and this project's README, translated to our schema (Realm/Guild_Name,
years 717-718) and skipping anything that needs a chart (no `create_chart`
tool yet). Also includes two small-talk checks to confirm the agent does
NOT call the tool when it doesn't need to.

We're deliberately using only the one generic `execute_python_code` tool —
per the updated roadmap, dedicated tools per operation come later, only if
this generic approach proves insufficient somewhere.

In [13]:
REFERENCE_QUESTIONS = [
    "Which realm had the highest net income in 718?",
    "Which realm had the highest taxes in each quarter of 718?",
    "Which company had the highest operating income growth between quarters of 718?",
    "How has operating income evolved between 717 and 718, and what could be the reasons for this change?",
    "Hello!",
    "Thank you!",
]


def run_question(agent, question: str) -> None:
    """Run one question through the agent and print a compact transparency report.

    Args:
        agent: the compiled agent to invoke.
        question: the natural-language question to ask.
    """
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    print(f"Q: {question}")

    tool_calls_made = [
        call["args"]["code"]
        for message in result["messages"]
        if message.type == "ai"
        for call in message.tool_calls
    ]
    if tool_calls_made:
        print("Used execute_python_code with:")
        for code in tool_calls_made:
            print(f"  {code}")
    else:
        print("Answered directly, no tool call.")

    final_answer = result["messages"][-1].content
    print(f"A: {final_answer}")
    print("-" * 80)


for question in REFERENCE_QUESTIONS:
    run_question(agent, question)

Q: Which realm had the highest net income in 718?
Used execute_python_code with:
  print(df[df['Year'] == 718].groupby('Realm')['Net_Income'].sum().idxmax())
A: 

The realm with the highest net income in year 718 was **Garden of the Midnight Rose**.
--------------------------------------------------------------------------------
Q: Which realm had the highest taxes in each quarter of 718?
Used execute_python_code with:
  filtered_df = df[df['Year'] == 718]
result = filtered_df.groupby('Quarter').apply(lambda x: x.loc[x['Tax'].idxmax(), 'Realm'])
print(result)
A: 

For the year 718, the realm with the highest taxes in each quarter was:
- **Quarter 1**: Straits of the Bottled Storm
- **Quarter 2**: Grove of the Singing Palms
- **Quarter 3**: Garden of the Midnight Rose
- **Quarter 4**: Oasis of Whispering Sands
--------------------------------------------------------------------------------
Q: Which company had the highest operating income growth between quarters of 718?
Used execute_pyt

### Debug: raw AI messages for the two questions that returned empty

Q2 and Q3 both answered directly (no tool call) with empty visible content.
`finish_reason` tells us why: `"stop"` means the model genuinely finished
with nothing to show (unlikely but possible), `"length"` means it hit
`max_tokens` before producing visible text — almost certainly spent on
hidden `<think>...</think>` reasoning. We just raised `max_tokens` above;
this cell re-runs just these two questions and inspects the raw messages.

**Important:** re-running the `model = ChatOpenAI(...)` cell creates a new
model object, but `agent` was built from the *old* one. Re-run the
`create_agent(...)` cell too before running this, otherwise you're still
testing with the old `max_tokens=2048`.

In [13]:
DEBUG_QUESTIONS = [
    "Which realm had the highest taxes in each quarter of 718?",
    "Which company had the highest operating income growth between quarters of 718?",
]

for question in DEBUG_QUESTIONS:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    last_ai_message = [m for m in result["messages"] if m.type == "ai"][-1]

    print(f"Q: {question}")
    print("content:", repr(last_ai_message.content))
    print("tool_calls:", last_ai_message.tool_calls)
    print("finish_reason:", last_ai_message.response_metadata.get("finish_reason"))
    print("completion_tokens:", last_ai_message.response_metadata.get("token_usage", {}).get("completion_tokens"))
    print("-" * 80)

Q: Which realm had the highest taxes in each quarter of 718?
content: '\n\nFor the year 718, the realm with the highest taxes in each quarter was:\n\n- **Quarter 1**: Straights of the Bottled Storm  \n- **Quarter 2**: Grove of the Singing Palms  \n- **Quarter 3**: Garden of the Midnight Rose  \n- **Quarter 4**: Oasis of Whispering Sands  \n\nThis result identifies the realm with the maximum tax value in each quarter of 718.'
tool_calls: []
finish_reason: stop
completion_tokens: 821
--------------------------------------------------------------------------------
Q: Which company had the highest operating income growth between quarters of 718?
content: '\n\nThe company with the highest operating income growth between quarters of Year 718 is **Forty Thieves Foundry**. \n\nThis was determined by calculating the difference in operating income between consecutive quarters for each guild in Year 718 and identifying the guild with the maximum growth.'
tool_calls: []
finish_reason: stop
complet

## Step 7: test loading an arbitrary user file

So far `df` was hardcoded to one specific CSV. This step checks the whole
pipeline generalizes to a *different* file with a *different* schema — a
stand-in for what a Streamlit file-upload widget will hand us later.

`bazaar_books/guild_ledger.csv` is a second synthetic dataset (same
fictional guilds, different metrics: market share, headcount, customer
satisfaction — no financial columns at all) to prove nothing here is
secretly tied to `Realm`/`Net_Income`/etc.

`load_table` picks CSV vs XLSX by file extension — the minimal version of
the future `read_new_table(path)` tool from the roadmap.

In [ ]:
def load_table(path: str) -> pd.DataFrame:
    """Load a user-provided table file into a DataFrame, by extension.

    Args:
        path: path to a .csv or .xlsx file.

    Returns:
        The loaded DataFrame.

    Raises:
        ValueError: if the file extension is neither .csv nor .xlsx.
    """
    if path.endswith(".csv"):
        return pd.read_csv(path)
    if path.endswith(".xlsx"):
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type for {path!r}, expected .csv or .xlsx")


# Reassign the global `df` — execute_python_code looks up `df` fresh from
# the global namespace on every call, so it automatically picks up whichever
# table is currently loaded without needing to be redefined.
df = load_table("bazaar_books/guild_ledger.csv")
print(df.shape)
print(df.dtypes)

# system_prompt and agent were built from the OLD df — rebuild both so the
# schema/preview and the agent's tool loop reflect the new table.
system_prompt = build_system_prompt(df)
agent = create_agent(
    model=model,
    tools=[execute_python_code],
    system_prompt=system_prompt,
)

In [ ]:
run_question(agent, "Which guild has the highest average customer satisfaction score?")

## Step 8: real user-driven file upload (not hardcoded, not a typed path)

Typing a path (previous attempt) is still not a real upload — it just moves
the "who decides the file" problem from code to a text prompt. A real
upload means: the user clicks a button, picks a file through their own
file system's dialog, and the file's bytes get handed to us — exactly how
Streamlit's `st.file_uploader` works.

`ipywidgets.FileUpload` is the Jupyter equivalent: it renders a clickable
button, opens your OS's native file picker, and gives us the raw bytes of
whatever you selected. Two cells, because the upload is asynchronous —
you need to click and pick a file in the first cell's widget before running
the second cell that reads what you picked.

In [20]:
import ipywidgets as widgets
from IPython.display import display

upload_widget = widgets.FileUpload(accept=".csv,.xlsx", multiple=False)
display(upload_widget)

# Click the button above, pick a .csv or .xlsx from your own filesystem,
# then run the next cell.

FileUpload(value=(), accept='.csv,.xlsx', description='Upload')

In [21]:
if not upload_widget.value:
    raise ValueError("No file selected yet — click the button above and choose a file first.")

uploaded_file = upload_widget.value[0]  # tuple of dicts in ipywidgets 8.x
filename = uploaded_file["name"]
file_bytes = io.BytesIO(uploaded_file["content"])

if filename.endswith(".csv"):
    df = pd.read_csv(file_bytes)
elif filename.endswith(".xlsx"):
    df = pd.read_excel(file_bytes)
else:
    raise ValueError(f"Unsupported file type: {filename}")

print(f"Loaded {filename} -> shape {df.shape}")
print(df.dtypes)

system_prompt = build_system_prompt(df)
agent = create_agent(
    model=model,
    tools=[execute_python_code],
    system_prompt=system_prompt,
)

print(
    "\n--- Data hygiene reminder ---\n"
    "Whatever file you just loaded is now baked into this cell's output.\n"
    "This notebook is version-controlled and gets pushed to a public repo:\n"
    "before committing, make sure the file above was a synthetic dataset\n"
    "(bazaar_books/*.csv), not real company data. If it wasn't, reload a\n"
    "synthetic file and re-run this cell, or clear this cell's output first."
)

Loaded caravan_accounts.csv -> shape (176, 9)
Realm                   str
Guild_Name              str
Year                  int64
Quarter               int64
Operating_Income    float64
EBITDA              float64
Tax                 float64
Net_Income          float64
GOGS                float64
dtype: object

--- Data hygiene reminder ---
Whatever file you just loaded is now baked into this cell's output.
This notebook is version-controlled and gets pushed to a public repo:
before committing, make sure the file above was a synthetic dataset
(bazaar_books/*.csv), not real company data. If it wasn't, reload a
synthetic file and re-run this cell, or clear this cell's output first.


### Test the agent on whatever you just uploaded

Generic financial question, phrased without assuming any specific column
names — the agent reads whatever schema `build_system_prompt` generated
for the file you picked in Step 8, and figures out the mapping itself
(same as it did for `Realm`/`Guild_Name` earlier in this notebook).

**Reminder:** if you uploaded a real file, this cell's output will contain
a real result. Do not commit it — before committing, re-run Step 8 with a
synthetic file (or clear this cell's output) so the notebook stays safe
to push.

In [22]:
run_question(agent, "What's the single most important insight you can find in this data?")

Q: What's the single most important insight you can find in this data?
Used execute_python_code with:
  print(df['Net_Income'].sum())
A: 

The total Net Income across all entries is **1,747,657.56**. This represents the cumulative profit after all expenses, taxes, and other deductions, highlighting the overall financial performance of the recorded guilds across the specified time period.
--------------------------------------------------------------------------------
